# 深市五个 E0：validation-only 投影验证
修改前基线为 34fe3e8 release，限价单已直接入簿；新版仅增加 E0 投影视图及成功成交原生序号审计，不改变恢复或生产输出。
每日期先对原值提取文件做前后二进制比较，再直接读取 /hdd/data/stock/raw_level2_parquet 做新版复验。日期/标的：20260320/300391、20260401/001257+301683、20260706/001248、20260806/001232。不是全市场标的验收。
完整命令、二进制 SHA256、时间和退出码保存在 reports/20260907-sz-e0-projection-verification/*-run.json。原始与提取验证记录仅允许日级属性来源路径不同。
生产 30s 截面和最终完整收盘盘口的前后 Parquet 字节相等检查保存在 production-output-audit.json；参考 E0 未被过滤。
诊断提取器首次遗漏 Parquet 独立 footer 元数据时被 schema 校验拒绝，失败产物保留在 target/e0-fixture-metadata-failed，不纳入验收；修正提取器后执行下列最终验证。

In [ ]:
from pathlib import Path
import json, runpy, hashlib
root = Path.cwd()
if root.name == 'analysis':
    root = root.parent
checks = runpy.run_path(str(root / 'analysis/verify_sz_e0_projection.py'))
result = checks['audit']()
out = checks['OUT']
receipt = json.loads((out / 'production-output-audit.json').read_text())
files = [out / f'replay-output-{label}/date=20260320/market=SZ/channel=2014/part-0.parquet' for label in ['before', 'after']]
assert all(hashlib.sha256(path.read_bytes()).hexdigest() == receipt['sha256'] for path in files)
final = json.loads((out / 'final-binary-audit.json').read_text())
assert final['all_four_date_reports_identical'] and final['production_output_byte_identical']
for date, _ in checks['CASES']:
    assert json.loads((out / f'{date}-after-final-extract.json').read_text()) == json.loads((out / f'{date}-after-extract.json').read_text())
[(r['date'], r['symbols'], r['matched'], r['mismatched']) for r in result]